In [5]:
import os
import struct
import re

TARGET_WEBP = "test_files/from_web/857f75db04db84340d4efa53061ca26fc7c7e6f65f80899c57f8d65b4c9d5678_recovered.webp"

print("=" * 60)
print(f"STAGE 1: RIFF Structure & Overlay Audit -> {os.path.basename(TARGET_WEBP)}")
print("=" * 60)

file_size = os.path.getsize(TARGET_WEBP)
print(f"File size on disk: {file_size:,} bytes")

with open(TARGET_WEBP, "rb") as f:
    header = f.read(12)
    if len(header) < 12 or header[:4] != b"RIFF" or header[8:] != b"WEBP":
        raise ValueError("Invalid file: missing RIFF/WEBP container signature.")
    
    declared_riff_len = struct.unpack("<I", header[4:8])[0]
    total_container_len = declared_riff_len + 8
    diff = file_size - total_container_len
    
    print(f"Declared RIFF size: {total_container_len:,} bytes")
    if diff > 0:
        print(f"[!] ANOMALY: {diff:,} trailing bytes detected past RIFF container!")
        f.seek(total_container_len)
        trailing_preview = f.read(min(diff, 64))
        print(f"    Trailing Hex: {trailing_preview.hex(' ')}")
        with open("test_files/carved_webp_overlay.bin", "wb") as out:
            f.seek(total_container_len)
            out.write(f.read())
        print("    Saved overlay to test_files/carved_webp_overlay.bin")
    else:
        print("[+] Clean container: no trailing overlay bytes.")
        
    # Parse chunk hierarchy
    f.seek(12)
    chunk_counts = {}
    standard_chunks = {"VP8 ", "VP8L", "VP8X", "ANIM", "ANMF", "ALPH", "ICCP", "EXIF", "XMP "}
    
    while True:
        chunk_hdr = f.read(8)
        if len(chunk_hdr) < 8:
            break
        tag = chunk_hdr[:4].decode("latin1", errors="ignore")
        size = struct.unpack("<I", chunk_hdr[4:8])[0]
        chunk_counts[tag] = chunk_counts.get(tag, 0) + 1
        
        # Extract metadata chunks or unknown custom boxes
        if tag in ("EXIF", "XMP ") or tag not in standard_chunks:
            chunk_data = f.read(size)
            print(f"\n[i] Extracted Chunk [{tag}] at offset {f.tell() - size - 8} ({size:,} bytes)")
            print(f"    Preview: {''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk_data[:64])}")
            tokens = re.findall(rb"[A-Za-z0-9_\-\{\}]{5,}", chunk_data)
            if tokens:
                print(f"    Tokens: {[t.decode(errors='ignore') for t in tokens[:6]]}")
            # Pad seek
            if size % 2 != 0:
                f.seek(1, 1)
        else:
            # Skip standard video bitstream chunks
            skip_len = size + (size % 2)
            f.seek(skip_len, 1)

print(f"\nChunk Summary: {chunk_counts}")

STAGE 1: RIFF Structure & Overlay Audit -> 857f75db04db84340d4efa53061ca26fc7c7e6f65f80899c57f8d65b4c9d5678_recovered.webp
File size on disk: 241,083,862 bytes
Declared RIFF size: 241,083,862 bytes
[+] Clean container: no trailing overlay bytes.

[i] Extracted Chunk [EXIF] at offset 241083818 (36 bytes)
    Preview: II*.......1...............ezgif.com.
    Tokens: ['ezgif']

Chunk Summary: {'VP8X': 1, 'ANIM': 1, 'ANMF': 12267, 'EXIF': 1}


In [2]:
import numpy as np
from PIL import Image

print("=" * 60)
print("STAGE 2: Animation Timeline & Bitplane Density Profile")
print("=" * 60)

with Image.open(TARGET_WEBP) as im:
    n_frames = getattr(im, "n_frames", 1)
    width, height = im.size
    print(f"Dimensions: {width}x{height} | Mode: {im.mode} | Frames: {n_frames:,}")
    
    # 1. Sample timeline to monitor pixel entropy
    sample_count = min(15, n_frames)
    step = max(1, n_frames // sample_count)
    checkpoints = [i * step for i in range(sample_count)]
    
    print(f"\n{'Frame':>8} | {'Mean RGB':>10} | {'LSB Density (1s)':>18} | {'Active Bytes':>15}")
    print("-" * 58)
    for idx in checkpoints:
        im.seek(idx)
        arr = np.array(im.convert("RGB"))
        flat = arr.flatten()
        lsb = flat & 1
        density = float(np.mean(lsb))
        packed = np.packbits(lsb)
        active = int(np.count_nonzero(packed))
        print(f"{idx:>8,} | {float(np.mean(arr)):>10.1f} | {density:>17.3%} | {active:>15,}")

    # 2. Check temporal LSB across frames (if multi-frame)
    if n_frames > 16:
        test_count = min(n_frames, 256)
        temporal_bits = []
        for idx in range(test_count):
            im.seek(idx)
            frame = np.array(im.convert("RGB"))
            temporal_bits.extend(frame[0, 0, :] & 1)
        
        temporal_bytes = np.packbits(temporal_bits[:len(temporal_bits) - (len(temporal_bits) % 8)])
        print(f"\nTemporal LSB probe ({test_count} frames): {bytes(temporal_bytes[:16]).hex(' ')}")

STAGE 2: Animation Timeline & Bitplane Density Profile
Dimensions: 640x640 | Mode: RGB | Frames: 12,267

   Frame |   Mean RGB |   LSB Density (1s) |    Active Bytes
----------------------------------------------------------
       0 |      230.7 |            3.736% |          11,637
     817 |      231.0 |            4.095% |          12,555
   1,634 |      230.4 |            4.058% |          12,620
   2,451 |      223.0 |            5.072% |          15,725
   3,268 |      223.2 |            5.295% |          16,378
   4,085 |      225.0 |            4.728% |          14,633
   4,902 |      230.7 |            3.797% |          11,675
   5,719 |      231.0 |            4.067% |          12,493
   6,536 |      230.9 |            4.103% |          12,775
   7,353 |      223.0 |            5.096% |          15,745
   8,170 |      223.2 |            5.358% |          16,440
   8,987 |      226.1 |            4.702% |          14,764
   9,804 |      230.7 |            3.779% |          11

In [3]:
import re
import numpy as np
from PIL import Image

print("=" * 60)
print("STAGE 3: Spatial LSB Steganography Extraction")
print("=" * 60)

MAGIC_SIGNATURES = {
    b"PK\x03\x04": "ZIP archive",
    b"7z\xbc\xaf\x27\x1c": "7-Zip archive",
    b"\x1f\x8b": "GZIP archive",
    b"BZh": "BZIP2 archive",
    b"%PDF": "PDF document",
    b"\x7fELF": "Linux ELF executable",
    b"MZ": "Windows PE executable"
}

def inspect_payload(raw_bytes, context_label):
    for magic, name in MAGIC_SIGNATURES.items():
        if raw_bytes.startswith(magic):
            print(f"[!] HIT: Embedded {name} located in {context_label}!")
            out_name = f"test_files/extracted_{context_label.replace(' ', '_').lower()}.bin"
            with open(out_name, "wb") as out:
                out.write(raw_bytes)
            print(f"    Saved payload to {out_name}")
            return True
            
    # Scan for plain ASCII text or flag formats
    tokens = re.findall(rb"[A-Za-z0-9_\-\{\}]{6,}", raw_bytes[:1024])
    printable_ratio = sum(32 <= b < 127 for b in raw_bytes[:256]) / 256.0
    
    if printable_ratio > 0.55 or any(b"flag" in t.lower() or b"ctf" in t.lower() for t in tokens):
        print(f"[!] High printable ASCII density ({printable_ratio:.1%}) in {context_label}")
        print("    ASCII Preview:", "".join(chr(b) if 32 <= b < 127 else "." for b in raw_bytes[:80]))
        if tokens:
            print("    Tokens:", [t.decode(errors="ignore") for t in tokens[:6]])
        return True
    return False

with Image.open(TARGET_WEBP) as im:
    n_frames = getattr(im, "n_frames", 1)
    # Prioritize first frames and the final frame
    frames_to_test = sorted(list(set([0, 1, 2, n_frames - 1])))
    hit_found = False
    
    for f_idx in frames_to_test:
        if f_idx >= n_frames:
            continue
        im.seek(f_idx)
        rgb = np.array(im.convert("RGB"))
        
        # Interleaved RGB LSB
        packed_all = np.packbits(rgb.flatten() & 1).tobytes()
        if inspect_payload(packed_all, f"Frame #{f_idx} Interleaved RGB"):
            hit_found = True
            break
            
        # Planar channel LSB (Red, Green, Blue isolated)
        for c_idx, c_name in enumerate(["Red", "Green", "Blue"]):
            packed_ch = np.packbits(rgb[:, :, c_idx].flatten() & 1).tobytes()
            if inspect_payload(packed_ch, f"Frame #{f_idx} {c_name} Channel"):
                hit_found = True
                break
        if hit_found:
            break

    if not hit_found:
        print("[+] No structured steganographic payloads or magic signatures in tested bitplanes.")

STAGE 3: Spatial LSB Steganography Extraction
[+] No structured steganographic payloads or magic signatures in tested bitplanes.


In [4]:
import numpy as np
from PIL import Image

print("=" * 60)
print("STAGE 4: Animation Flash Card / Subliminal Frame Detector")
print("=" * 60)

with Image.open(TARGET_WEBP) as im:
    n_frames = getattr(im, "n_frames", 1)
    
    if n_frames <= 1:
        print("[i] Static WebP image. Animation flash card analysis skipped.")
    else:
        # Adaptive subsampling: scan every frame if short, stride if massive
        stride = 1 if n_frames < 300 else max(2, n_frames // 2000)
        print(f"Scanning {n_frames:,} frames (stride={stride})...")
        
        dark_counts = []
        for idx in range(0, n_frames, stride):
            im.seek(idx)
            arr = np.array(im.convert("L"))
            # Threshold dark pixels (text, line drawings, QR codes)
            dark_counts.append((idx, int(np.count_nonzero(arr < 200))))
            
        values = [c for _, c in dark_counts]
        median_val = float(np.median(values))
        std_val = float(np.std(values))
        print(f"Baseline dark pixels per frame: {median_val:.0f} (std: {std_val:.1f})")
        
        # Detect frames exceeding 4 standard deviations
        threshold = 4 * max(std_val, 400)
        outliers = [(idx, c) for idx, c in dark_counts if abs(c - median_val) > threshold]
        
        if outliers:
            print(f"\n[!] DETECTED {len(outliers)} anomalous outlier frame(s):")
            for idx, count in outliers[:10]:
                print(f"  --> Frame #{idx}: {count:,} dark pixels (baseline: {median_val:.0f})")
                im.seek(idx)
                im.convert("RGB").save(f"test_files/outlier_frame_{idx}.png")
            print("    Exported anomalous frames to test_files/outlier_frame_*.png")
        else:
            print("[+] Clean visual timeline: No visual flash cards or subliminal anomaly frames.")

STAGE 4: Animation Flash Card / Subliminal Frame Detector
Scanning 12,267 frames (stride=6)...
Baseline dark pixels per frame: 49736 (std: 6390.2)
[+] Clean visual timeline: No visual flash cards or subliminal anomaly frames.
